# UDP RTT Benchmark

Thin notebook wrapper around the terminal RTT benchmark.

This runs the same `run_pynq_rtt.py` command you would use in the terminal, saves JSON/CSV, and displays the key stats clearly in Jupyter.

What it measures:
- direct UDP round-trip time from board to EC2 server and back
- dedicated RTT probe packets on server port `9000`

What it does not measure:
- monitor/browser latency
- full button-to-visible gameplay latency

Most useful stat for the report:
- `p95_rtt_ms`

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

NOTEBOOK_DIR = Path.cwd()
SCRIPT_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / "run_pynq_rtt.py").exists() else NOTEBOOK_DIR / "pynq_client_tests"


## Configure Run

In [ ]:
SERVER = "3.9.71.204"
PORT = 9000
LABEL = "idle"
SAMPLES = 100
TIMEOUT_S = 1.0
CSV_PATH = Path(f"udp_rtt_{LABEL}.csv")
JSON_PATH = Path(f"udp_rtt_{LABEL}.json")

SERVER, PORT, LABEL, SAMPLES, TIMEOUT_S, CSV_PATH, JSON_PATH

## Run Benchmark

This runs the same command you would use in the terminal.

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_DIR / "run_pynq_rtt.py"),
    "--server", SERVER,
    "--port", str(PORT),
    "--label", LABEL,
    "--samples", str(SAMPLES),
    "--timeout", str(TIMEOUT_S),
    "--csv-out", str(CSV_PATH),
    "--json-out", str(JSON_PATH),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## Load Stats

In [ ]:
report = json.loads(JSON_PATH.read_text())
report

## Key Metrics

In [ ]:
summary = {
    "label": report.get("label"),
    "samples_ok": report.get("samples_ok"),
    "samples_lost": report.get("samples_lost"),
    "loss_pct": report.get("loss_pct"),
    "avg_rtt_ms": (report.get("rtt_stats") or {}).get("avg_ms"),
    "p50_rtt_ms": (report.get("rtt_stats") or {}).get("p50_ms"),
    "p95_rtt_ms": (report.get("rtt_stats") or {}).get("p95_ms"),
    "max_rtt_ms": (report.get("rtt_stats") or {}).get("max_ms"),
    "min_rtt_ms": (report.get("rtt_stats") or {}).get("min_ms"),
    "stddev_rtt_ms": (report.get("rtt_stats") or {}).get("stddev_ms"),
}
summary